## Detecting outperformers - Data preparation

In [4]:
## Channel data
import pandas as pd
from src.data.dataloader import load_channels_dataset, load_video_metadata


In [5]:
channel_df = load_channels_dataset()

In [6]:
print(len(channel_df))
print(channel_df.describe())
channel_df.head(10)


136470
       subscribers_cc      videos_cc  subscriber_rank_sb        weights
count    1.364700e+05  136470.000000        1.364700e+05  136470.000000
mean     2.466022e+05     699.784231        3.570087e+05       7.349363
std      1.150096e+06    4524.980928        2.711139e+05       5.277328
min      1.000000e+04      10.000000        3.000000e+00       2.086500
25%      1.888575e+04      70.000000        1.161305e+05       4.231500
50%      4.240000e+04     175.000000        3.015670e+05       6.444000
75%      1.370000e+05     443.000000        5.704942e+05       9.243500
max      1.121395e+08  461923.000000        1.030844e+06      53.150500


,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087
1,Education,2006-09-01,UCbCmjCuTUZos6Inko4u57UQ,Cocomelon - Nursery ...,60100000,458,7.0,2.087
2,Entertainment,2006-09-20,UCpEhnqL0y41EpW2TvWAHD7Q,SET India,56018869,32661,8.0,2.087
3,Howto & Style,2016-11-15,UC295-Dw_tDNtZXFeAPAW6Aw,5-Minute Crafts,60600000,3591,9.0,2.087
4,Sports,2007-05-11,UCJ5v_MCY6GNUBTO8-D3XoAg,WWE,48400000,43421,11.0,2.087
5,Entertainment,2007-01-15,UCIwFjwMjI0y7PDBVEO9-bkQ,Justin Bieber,46574085,134,12.0,2.087
6,Music,2014-03-12,UCFFbwnve3yF62-tVXkTyHqg,Zee Music Company,43451109,4241,13.0,2.087
7,Sports,2009-03-17,UCRijo3ddMTht_IHyNSNXpNQ,Dude Perfect,45800000,213,14.0,2.087
8,Entertainment,2015-05-12,UCk8GzjMOrta8yxDcKfylJYw,✿ Kids Diana Show,34700000,599,16.0,2.087
9,Music,2006-08-08,UC0C-w0YjGpqDXGB8IHb662A,Ed Sheeran,42000000,154,17.0,2.087


In [9]:
channel_df.columns

Index(['category_cc', 'join_date', 'channel', 'name_cc', 'subscribers_cc',
       'videos_cc', 'subscriber_rank_sb', 'weights'],
      dtype='object')

In [1]:
from src.data.dataloader import iterate_videos_metadata_stream
from src.data.preprocessing import videos_preprocessing
from collections import defaultdict
from tqdm import tqdm

import pandas as pd
import numpy as np
import gc

In [3]:
# initialize accumulators
# ============================
    #This part takes 36 minutes
#=============================



def reset_accumulators():
    return (
        defaultdict(float),  # efficiency sum
        defaultdict(int),    # efficiency count
        defaultdict(float),  # views sum
        defaultdict(float),  # duration sum
        defaultdict(int)     # video count
    )

(
    channel_efficiency_sum,
    channel_efficiency_count,
    channel_views_sum,
    channel_duration_sum,
    channel_video_count
) = reset_accumulators()

row_count = 0
part_id = 0
rows_since_flush = 0
FLUSH_INTERVAL = 10_000_000  # flush every 10M videos

# --- use the new streaming iterator here ---
for chunk_video_df in tqdm(iterate_videos_metadata_stream(
                                       window_size=1_000_000,
                                       start_line=0),
        desc="Loading videos"):

    chunk_video_df = videos_preprocessing(chunk_video_df)
    if "channel_id" in chunk_video_df.columns:
        grouped_chunk = (
            chunk_video_df.groupby("channel_id")[["engagement_efficiency", "view_count", "duration"]]
            .agg(["sum", "count"])
        )
        grouped_chunk.columns = ["_".join(c) for c in grouped_chunk.columns]
        grouped_chunk = grouped_chunk.reset_index()

        for row in grouped_chunk.itertuples(index=False):
            ch = row.channel_id
            channel_efficiency_sum[ch] += row.engagement_efficiency_sum
            channel_efficiency_count[ch] += row.engagement_efficiency_count
            channel_views_sum[ch] += row.view_count_sum
            channel_duration_sum[ch] += row.duration_sum
            channel_video_count[ch] += row.duration_count

    row_count += len(chunk_video_df)
    rows_since_flush += len(chunk_video_df)

    # --- periodic flush ---
    if rows_since_flush >= FLUSH_INTERVAL:
        records = []
        for ch in channel_video_count.keys():
            n_v = channel_video_count[ch]
            if n_v == 0:
                continue
            records.append({
                "channel_id": ch,
                "avg_efficiency": channel_efficiency_sum[ch] / channel_efficiency_count[ch]
                    if channel_efficiency_count[ch] > 0 else np.nan,
                "avg_views": channel_views_sum[ch] / n_v,
                "avg_duration": channel_duration_sum[ch] / n_v,
                "video_count": n_v
            })
        part_df = pd.DataFrame(records)
        part_df.to_parquet(f"cache/partial_stats_{part_id:02d}.parquet")
        print(f" Flushed partial {part_id} ({len(part_df):,} channels, {row_count:,} rows).")

        # reset accumulators
        (
            channel_efficiency_sum,
            channel_efficiency_count,
            channel_views_sum,
            channel_duration_sum,
            channel_video_count
        ) = reset_accumulators()
        rows_since_flush = 0
        part_id += 1
        gc.collect()

# --- flush last batch ---
if rows_since_flush > 0:
    records = []
    for ch in channel_video_count.keys():
        n_v = channel_video_count[ch]
        if n_v == 0:
            continue
        records.append({
            "channel_id": ch,
            "avg_efficiency": channel_efficiency_sum[ch] / channel_efficiency_count[ch]
                if channel_efficiency_count[ch] > 0 else np.nan,
            "avg_views": channel_views_sum[ch] / n_v,
            "avg_duration": channel_duration_sum[ch] / n_v,
            "video_count": n_v
        })
    part_df = pd.DataFrame(records)
    part_df.to_parquet(f"cache/partial_stats_{part_id:02d}.parquet")
    print(f" Flushed final partial {part_id} ({len(part_df):,} channels, {row_count:,} rows).")

print(f"\n____ Done reading {row_count:,} rows total.")

Loading videos: 1it [00:26, 26.94s/it]

++ Chunk 000: lines 0–1,000,000 | total=1,000,000


Loading videos: 2it [00:54, 27.56s/it]

++ Chunk 001: lines 1,000,000–2,000,000 | total=2,000,000


Loading videos: 3it [01:21, 27.22s/it]

++ Chunk 002: lines 2,000,000–3,000,000 | total=3,000,000


... (138 outputs left out)

Loading videos: 73it [36:19, 29.14s/it]

++ Final chunk: lines 72,000,000–EOF | rows=924,794


Loading videos: 73it [36:19, 29.86s/it]

 Flushed final partial 7 (4,861 channels, 72,924,794 rows).

____ Done reading 72,924,794 rows total.


In [10]:
import glob

parquet_files = sorted(glob.glob("cache/partial_stats_*.parquet"))
print(f"Found {len(parquet_files)} partials:", parquet_files)

parts = [pd.read_parquet(f) for f in parquet_files]
video_stats_df = pd.concat(parts, ignore_index=True)

#Combines duplicates - channels that appear in multiple partials
video_stats_df = (
    video_stats_df
    .groupby("channel_id")[["avg_efficiency", "avg_views", "avg_duration", "video_count"]]
    .mean()
    .reset_index()
)
print("Aggregated video_stats_df shape:", video_stats_df.shape)

Found 8 partials: ['cache\\partial_stats_00.parquet', 'cache\\partial_stats_01.parquet', 'cache\\partial_stats_02.parquet', 'cache\\partial_stats_03.parquet', 'cache\\partial_stats_04.parquet', 'cache\\partial_stats_05.parquet', 'cache\\partial_stats_06.parquet', 'cache\\partial_stats_07.parquet']
Aggregated video_stats_df shape: (136470, 5)


In [12]:
channels_final = channel_df.merge(
    video_stats_df,
    left_on="channel", right_on="channel_id",
    how="left"
)

print(len(channels_final))
print(channels_final.describe())
channels_final.head(10)




136470
       subscribers_cc      videos_cc  subscriber_rank_sb        weights  \
count    1.364700e+05  136470.000000        1.364700e+05  136470.000000   
mean     2.466022e+05     699.784231        3.570087e+05       7.349363   
std      1.150096e+06    4524.980928        2.711139e+05       5.277328   
min      1.000000e+04      10.000000        3.000000e+00       2.086500   
25%      1.888575e+04      70.000000        1.161305e+05       4.231500   
50%      4.240000e+04     175.000000        3.015670e+05       6.444000   
75%      1.370000e+05     443.000000        5.704942e+05       9.243500   
max      1.121395e+08  461923.000000        1.030844e+06      53.150500   

       avg_efficiency     avg_views   avg_duration    video_count  
count   136466.000000  1.364700e+05  136470.000000  136470.000000  
mean         0.028732  4.034839e+05     696.417359     534.139844  
std          0.031729  3.148789e+06    1179.114630    1861.587003  
min          0.000000  0.000000e+00       1.0

,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights,channel_id,avg_efficiency,avg_views,avg_duration,video_count
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,UC-lHJZR3Gqxm24_Vd_AJ5Yw,0.037864,5.971857e+06,878.499873,3927.0
1,Education,2006-09-01,UCbCmjCuTUZos6Inko4u57UQ,Cocomelon - Nursery ...,60100000,458,7.0,2.087,UCbCmjCuTUZos6Inko4u57UQ,0.001860,9.358046e+07,1123.391775,462.0
2,Entertainment,2006-09-20,UCpEhnqL0y41EpW2TvWAHD7Q,SET India,56018869,32661,8.0,2.087,UCpEhnqL0y41EpW2TvWAHD7Q,0.004200,1.705198e+06,1276.033720,23903.0
3,Howto & Style,2016-11-15,UC295-Dw_tDNtZXFeAPAW6Aw,5-Minute Crafts,60600000,3591,9.0,2.087,UC295-Dw_tDNtZXFeAPAW6Aw,0.009845,4.404196e+06,933.562086,3624.0
4,Sports,2007-05-11,UCJ5v_MCY6GNUBTO8-D3XoAg,WWE,48400000,43421,11.0,2.087,UCJ5v_MCY6GNUBTO8-D3XoAg,0.013796,8.939673e+05,194.059854,40549.0
5,Entertainment,2007-01-15,UCIwFjwMjI0y7PDBVEO9-bkQ,Justin Bieber,46574085,134,12.0,2.087,UCIwFjwMjI0y7PDBVEO9-bkQ,0.022880,4.598778e+06,138.257812,128.0
6,Music,2014-03-12,UCFFbwnve3yF62-tVXkTyHqg,Zee Music Company,43451109,4241,13.0,2.087,UCFFbwnve3yF62-tVXkTyHqg,0.010053,4.911759e+06,338.364786,4271.0
7,Sports,2009-03-17,UCRijo3ddMTht_IHyNSNXpNQ,Dude Perfect,45800000,213,14.0,2.087,UCRijo3ddMTht_IHyNSNXpNQ,0.011369,4.273825e+07,395.719626,214.0
8,Entertainment,2015-05-12,UCk8GzjMOrta8yxDcKfylJYw,✿ Kids Diana Show,34700000,599,16.0,2.087,UCk8GzjMOrta8yxDcKfylJYw,0.001259,2.294375e+07,504.805648,602.0
9,Music,2006-08-08,UC0C-w0YjGpqDXGB8IHb662A,Ed Sheeran,42000000,154,17.0,2.087,UC0C-w0YjGpqDXGB8IHb662A,0.018128,1.172810e+08,314.383117,154.0


In [14]:
# Saving this dataset for later use
from src.utils.paths import DATA_DIR
import os

channels_final.to_csv(os.path.join(DATA_DIR, "channels_with_efficiency.tsv.gz"),
                      sep="\t",      # tab-separated values
                      index=False,   # don’t include DataFrame index
                      compression="gzip")